In [1]:
# Install fine-tuning dependencies (run once).
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 115.9 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00:00:0100:01


In [3]:
# Secure Hugging Face login in notebook (token is not printed).
from getpass import getpass
from huggingface_hub import login, whoami
from huggingface_hub.errors import HfHubHTTPError

hf_token = getpass("Enter your Hugging Face USER token (starts with hf_): ").strip()
if not hf_token.startswith("hf_"):
    raise ValueError("Invalid token format. Use a personal Hugging Face token that starts with 'hf_'.")

try:
    login(token=hf_token, add_to_git_credential=False)
    me = whoami(token=hf_token)
    print(f"Hugging Face login successful as: {me['name']}")
except HfHubHTTPError as e:
    raise RuntimeError(
        "Login failed. Use a PERSONAL token (not an organization token) with Read access. "
        "Also confirm your token is active in HF settings. Original error: " + str(e)
    )

Hugging Face login successful as: babayaaaaga


In [6]:
import torch
MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "mlabonne/guanaco-llama2-1k"
DATASET_SPLIT = "train"
TEXT_FIELD = "text"  # Guanaco dataset field

OUTPUT_DIR = "./llama2-7b-lora-ft"
MAX_SEQ_LENGTH = 512
EPOCHS = 1
LEARNING_RATE = 2e-4
USE_4BIT = True

def recommend_batching():
    if not torch.cuda.is_available():
        # CPU fallback
        return 1, 16

    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if total_gb < 10:
        return 1, 32
    if total_gb < 16:
        return 2, 16
    if total_gb < 24:
        return 4, 8
    return 8, 4

BATCH_SIZE, GRAD_ACCUM_STEPS = recommend_batching()
print(f"Auto batch config -> per_device_batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM_STEPS}")

Auto batch config -> per_device_batch=2, grad_accum=16


In [7]:

import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("Warning: fine-tuning on CPU is very slow. A CUDA GPU is strongly recommended.")

Device: cuda


In [8]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = None
if USE_4BIT and torch.cuda.is_available():
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16,
 )

# Better memory behavior for QLoRA training.
model.config.use_cache = False
model.gradient_checkpointing_enable()
print("Model and tokenizer loaded.")

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model and tokenizer loaded.


In [9]:

raw_dataset = load_dataset(DATASET_ID, split=DATASET_SPLIT)

candidate_fields = [TEXT_FIELD, "text", "prompt", "content", "quote", "instruction"]
selected_field = next((f for f in candidate_fields if f and f in raw_dataset.column_names), None)
if selected_field is None:
    raise ValueError(f"Could not infer text column. Available columns: {raw_dataset.column_names}")

def to_text(example):
    # Support both plain text datasets and prompt/completion style rows.
    if "prompt" in example and "completion" in example:
        return {"text": f"{example['prompt']}\n{example['completion']}"}
    return {"text": str(example[selected_field])}

train_dataset = raw_dataset.map(to_text, remove_columns=raw_dataset.column_names)
print(f"Using text field: {selected_field}")
print(train_dataset[0])
print(f"Training rows: {len(train_dataset)}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Using text field: text
{'text': '<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'}
Training rows: 1000


In [10]:

from huggingface_hub import model_info, dataset_info
from huggingface_hub.errors import HfHubHTTPError

if MODEL_ID.startswith("REPLACE_WITH"):
    raise ValueError("Set MODEL_ID in Cell 3 before continuing.")

try:
    _ = model_info(MODEL_ID, token=hf_token)
    _ = dataset_info(DATASET_ID, token=hf_token)
except HfHubHTTPError as e:
    raise RuntimeError(
        "Preflight failed. Ensure your token has Read permission and you have accepted gated model terms for "
        f"{MODEL_ID} on Hugging Face. Original error: {e}"
    )

print(f"Model access verified: {MODEL_ID}")
print(f"Dataset access verified: {DATASET_ID} [{DATASET_SPLIT}]")

Model access verified: meta-llama/Llama-2-7b-hf
Dataset access verified: mlabonne/guanaco-llama2-1k [train]


In [ ]:

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_strategy="epoch",
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()
print("Fine-tuning complete.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.418066


In [ ]:

adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved to: {adapter_dir}")

